# ACC 380K — Notebook 8
## Pandas foundations: tables and selection

**Notebook version 0.10.1 · October 13, 2026**

**Pandas** is the Python package most analysts use for tables. A Pandas table is called a **DataFrame**. This notebook teaches a small set of moves that you will use in every later notebook and in Case 1: say what one row means, pick columns, keep the rows you want, work on a copy, and stack several files into one table.

It starts with three rows typed by hand, then downloads the SEC's list of every filing made from October to December 2025, about 277,000 rows. It ends with nested loops that download every quarter from 2015 to 2025.

In Colab, choose **File → Save a copy in Drive** and work in your own copy.


## 12:30 Section — live class record

This copy began from **Starter**. Nick edits it during the 12:30 class and saves it at the end. Use Starter when you want a fresh copy of the exercise.


## What you should be able to do

- Build a small DataFrame and read its shape, columns, and column types.
- Download a public file from the SEC's EDGAR system, save it, and read it with `pd.read_csv`.
- Say what one row of a table represents (its **grain**) and check whether an ID column is unique.
- Select named columns with `.loc`, and rows by position with `.iloc`.
- Keep rows with a **Boolean mask**, including two conditions joined with `&` inside parentheses.
- Use `.copy()` so a new table does not change the original.
- Download many files with nested loops and stack them into one table with `pd.concat`.

Each numbered **Exercise** heading has a **Your turn** box: try it on your own or with your table. Run your work cell, then the collapsed **Check your work** cell below it. A line that starts with `NOT YET` tells you what to fix; revise your work cell and rerun both cells.

**Read → Predict → Run → Assess → Test**


## Read and select a table


## Packages first

Run this cell once each time you start a new Colab session. It loads the packages:

- `%pip install` installs **EdgarTools**, the package from Homework 3, and `import edgar` makes it available.
- `import pandas as pd` loads Pandas and lets us write `pd` for short. Every Pandas notebook starts with it.
- `import requests` loads the package that downloads files from the web.
- `from io import StringIO` lets Pandas read downloaded text as if it were a file.


In [1]:
%pip install -q edgartools==5.58.0
import edgar
import pandas as pd
import requests
from io import StringIO


In [3]:
import math as mth

In [4]:
mth.pi

3.141592653589793

## From a dictionary of lists to a table

In Notebook 5 you stored related values in a **dictionary**: each key names something, and its value can be a **list**. A DataFrame is built the same way. Each key becomes a **column name** and each list becomes that column's values. All the lists must have the same length, because position 0 in every list describes the same row.

The rows below describe three real reports Starbucks filed in the fall of 2025; you will find them again in an SEC file in a few minutes. `CIK` is the SEC's ID number for a company. An 8-K reports a major event, and the SEC numbers the kinds of event as **items**: `2.02` is quarterly results, `7.01` is information shared with investors, and `9.01` means documents are attached. So one value in the `Items` column is a whole list of item numbers. A 10-K is an annual report and has no 8-K items, so its list is empty.

**Predict:** How many rows and how many columns will the table have?


In [5]:
data_dict = {
    "CIK": [829224, 829224, 829224],
    "Form Type": ["8-K", "8-K", "10-K"],
    "Date Filed": ["2025-10-29", "2025-11-03", "2025-11-14"],
    "Items": [["2.02", "9.01"], ["7.01", "9.01"], []]
}


In [8]:
starbucks = pd.DataFrame(data_dict)


In [9]:
starbucks


,CIK,Form Type,Date Filed,Items
0,829224,8-K,2025-10-29,"[2.02, 9.01]"
1,829224,8-K,2025-11-03,"[7.01, 9.01]"
2,829224,10-K,2025-11-14,[]


Colab shows a DataFrame as a table when it is the last line of a cell. The numbers 0 to 2 on the left are the **row labels**, also called the **index**. Pandas created them for us.

| Code | What it gives |
|---|---|
| `starbucks.shape` | a tuple `(number of rows, number of columns)` |
| `starbucks.columns.tolist()` | the column names, as a list |
| `starbucks.dtypes` | each column's type: `int64` for whole numbers, `float64` for decimals, `object` for text and other Python values such as lists |
| `starbucks["Form Type"]` | one column |
| `starbucks.loc[2, "Form Type"]` | one value, chosen by **label**: the row labeled 2, the column named `Form Type` |
| `starbucks.iloc[2, 1]` | one value, chosen by **position**: the third row, the second column (positions count from 0, as in a list) |

**Predict:** What will each of the next six cells show? Then run them one at a time.


In [10]:
starbucks.shape


(3, 4)

In [11]:
starbucks.columns.tolist()


['CIK', 'Form Type', 'Date Filed', 'Items']

In [12]:
starbucks.dtypes


,0
CIK,int64
Form Type,object
Date Filed,object
Items,object


In [17]:
starbucks[["CIK", "Form Type"]]


,CIK,Form Type
0,829224,8-K
1,829224,8-K
2,829224,10-K


In [18]:
starbucks


,CIK,Form Type,Date Filed,Items
0,829224,8-K,2025-10-29,"[2.02, 9.01]"
1,829224,8-K,2025-11-03,"[7.01, 9.01]"
2,829224,10-K,2025-11-14,[]


In [19]:
starbucks.loc[2, "Form Type"]


'10-K'

In [20]:
starbucks.iloc[2, 1]


'10-K'

`CIK` is `int64` because we typed whole numbers. `Form Type` and `Date Filed` are text, and `Items` holds a list in each row; Pandas labels both kinds `object`. Here the labels and the positions are the same numbers, so `.loc` and `.iloc` give the same value. That changes once we keep only some rows.

That is all the typing. Real tables are read from files.

## Real data: every SEC filing in one quarter

**EDGAR** is the SEC's filing system. For each quarter it publishes a **master index**: a text file that lists every filing made that quarter, one per line, with the five values separated by `|`. The index for October to December 2025 lists about 277,000 filings and is about 25 MB.

As in Homework 3, the SEC asks automated requests to identify who is making them. Run the next cell: if `sec_identity` is empty, it asks for your name and email. You can instead type them between the quotes. Use **your own** name and email; do not use someone else's identity or post a completed cell in a public repository.


In [21]:
sec_identity = ""
if sec_identity == "":
    name = input("Name: ")
    email = input("Email: ")
    sec_identity = name + " " + email
edgar.set_identity(sec_identity)
print("EDGAR identity set.")


Name: Nick Hallman
Email: nicholashallman@utexas.edu
EDGAR identity set.


### Download the file

`requests.get` asks the SEC's server for the file at `url`. The **User-Agent** header carries your identity, and `timeout=60` stops waiting after 60 seconds. The server answers with a **status code**: `200` means it sent the file, and `403` means it refused, usually because the identity is missing.


In [36]:
url = "https://www.sec.gov/Archives/edgar/full-index/2025/QTR4/master.idx"
#url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

response = requests.get(url, headers={"User-Agent": sec_identity}, timeout=60)


`response.content` holds the file as raw bytes. `.decode("latin-1")` turns the bytes into a string, using the character set EDGAR's index files are written in.


In [37]:
type(response.content)


bytes

In [38]:
# Convert the first 20 bytes into their raw integer values (0-255)
list(response.content[0:20])


[68,
 101,
 115,
 99,
 114,
 105,
 112,
 116,
 105,
 111,
 110,
 58,
 32,
 32,
 32,
 32,
 32,
 32,
 32,
 32]

In [39]:
text = response.content.decode("latin-1")
type(text)


str

In [40]:
len(text)

25123884

In [41]:
print(text[0:1000])


Description:           Master Index of EDGAR Dissemination Feed
Last Data Received:    December 31, 2025
Comments:              webmaster@sec.gov
Anonymous FTP:         ftp://ftp.sec.gov/edgar/
Cloud HTTP:            https://www.sec.gov/Archives/




CIK|Company Name|Form Type|Date Filed|Filename
--------------------------------------------------------------------------------
1000045|OLD MARKET CAPITAL Corp|10-Q|2025-11-14|edgar/data/1000045/0001437749-25-035199.txt
1000045|OLD MARKET CAPITAL Corp|25|2025-12-22|edgar/data/1000045/0001437749-25-038479.txt
1000045|OLD MARKET CAPITAL Corp|4|2025-10-08|edgar/data/1000045/0001398344-25-019020.txt
1000045|OLD MARKET CAPITAL Corp|4|2025-10-08|edgar/data/1000045/0001578005-25-000002.txt
1000045|OLD MARKET CAPITAL Corp|4|2025-10-08|edgar/data/1000045/0001689627-25-000002.txt
1000045|OLD MARKET CAPITAL Corp|4|2025-10-08|edgar/data/1000045/0001720104-25-000002.txt
1000045|OLD MARKET CAPITAL Corp|4|2025-10-08|edgar/data/1000045/0001887678-25-00000

In [35]:
# # Get the S&P 500 Sector and Company table
# df_sp500 = pd.read_html(StringIO(response.text))[0]

# df_sp500


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


The file starts with a few lines of description, then a line of column names, a line of dashes, and one line per filing.

### Save it to disk

`open(local_filename, "wb")` opens a file for **w**riting **b**ytes, replacing any old file with that name; `.write` puts the bytes in, and `.close()` finishes the file. The cell saves only if the status code is 200, so an error page is never saved as data. In Colab, the file appears in the **Files** panel on the left.


In [ ]:
# response = requests.get("https://www.google.com/non-existent-page-demo-404", headers={"User-Agent": sec_identity}, timeout=60)
# response.status_code


In [43]:
local_filename = "master_2025_Q4.idx"

if response.status_code == 200:
    f = open(local_filename, "wb")
    f.write(response.content)
    f.close()
    print("Saved", local_filename)
else:
    print("Not saved: EDGAR answered with status code", response.status_code)
    print("What to do: check sec_identity (your name and email), rerun that cell, then rerun from the download cell.")


Saved master_2025_Q4.idx


### Read it with Pandas

`pd.read_csv` reads a file of separated values into a DataFrame. `sep="|"` says the values are separated by `|` instead of commas. `skiprows` lists the line numbers to skip, counting from 0: the description lines before the column names (lines 0 to 8) and the line of dashes after them (line 10). Line 9, the column names, becomes the header.

If this cell stops with `FileNotFoundError`, the file was not saved: read what the cell above printed.


In [49]:
filings = pd.read_csv("master_2025_Q4.idx", sep="|", skiprows = [0,1,2,3,4,5,6,7,8,10])


In [50]:
filings


,CIK,Company Name,Form Type,Date Filed,Filename
0,1000045,OLD MARKET CAPITAL Corp,10-Q,2025-11-14,edgar/data/1000045/0001437749-25-035199.txt
1,1000045,OLD MARKET CAPITAL Corp,25,2025-12-22,edgar/data/1000045/0001437749-25-038479.txt
2,1000045,OLD MARKET CAPITAL Corp,4,2025-10-08,edgar/data/1000045/0001398344-25-019020.txt
3,1000045,OLD MARKET CAPITAL Corp,4,2025-10-08,edgar/data/1000045/0001578005-25-000002.txt
4,1000045,OLD MARKET CAPITAL Corp,4,2025-10-08,edgar/data/1000045/0001689627-25-000002.txt
...,...,...,...,...,...
276617,99780,TRINITY INDUSTRIES INC,4,2025-10-02,edgar/data/99780/0000099780-25-000109.txt
276618,99780,TRINITY INDUSTRIES INC,4,2025-10-02,edgar/data/99780/0000099780-25-000110.txt
276619,99780,TRINITY INDUSTRIES INC,4,2025-10-02,edgar/data/99780/0000099780-25-000111.txt
276620,99780,TRINITY INDUSTRIES INC,8-K,2025-10-20,edgar/data/99780/0000099780-25-000114.txt


Each `Filename` ends with the filing's **accession number**, the SEC's ID for one filing, followed by `.txt`. A string can be sliced like a list: `[-24:-4]` keeps from 24 characters before the end up to, but not including, the last 4 (`.txt`).


In [53]:
filings.loc[0,"Filename"]

'edgar/data/1000045/0001437749-25-035199.txt'

In [54]:
filings.loc[0,"Filename"][-24:-4]


'0001437749-25-035199'

In [ ]:
filing_no = filings.loc[0,"Filename"][-24:-4]
print(filing_no)


With an accession number, EdgarTools from Homework 3 can fetch that one filing. `filing.document.markdown()` returns the filing's main document as text, and the third cell prints its first 1,000 characters. (This reuses the name `text`, which now holds the filing instead of the index.)


In [ ]:
filing = edgar.get_by_accession_number(filing_no)


In [ ]:
text = filing.document.markdown()


In [ ]:
print(text[0:1000])


Now imagine looping over `range(len(filings))` to pull each filing and do some analysis on it, store the result in a new data structure (e.g., a new dataframe, list, dictionary etc.) before moving onto the next filing...


## What does one row represent?

Before calculating anything, ask what one row represents. That is the table's **grain**.

| Column | Meaning |
|---|---|
| `CIK` | the SEC's ID for the company that filed |
| `Company Name` | the company's name, as the SEC records it |
| `Form Type` | the kind of report: `10-K` (annual report), `10-Q` (quarterly report), `8-K` (current report of a major event), `4` (an insider's trade), and many more |
| `Date Filed` | the day the SEC received it |
| `Filename` | where that filing's documents are stored on EDGAR |

`.is_unique` is `True` only if no value in a column repeats.

**Predict:** Will `CIK` be unique? Will `Filename` be unique? What type will `CIK` have?


In [ ]:
print("Shape:", filings.shape)
print()
print(filings.dtypes)
print()
print("CIK is unique:", filings["CIK"].is_unique)
print("Filename is unique:", filings["Filename"].is_unique)


- `CIK` repeats, because a company files many reports in a quarter. So one row is not one company.
- `pd.read_csv` saw only digits in `CIK`, so it read the column as whole numbers (`int64`), just as in our typed table. Compare it with a number, such as `829224`, without quotes. The other columns are text (`object`), including `Date Filed`.
- `Filename` is **almost** unique. In this quarter exactly one filing, by Anebulo Pharmaceuticals, is listed twice under two form types (`SC 13E3` and `SC TO-I`): one set of documents that served two purposes. Strictly, one row is one filing under one form type. For nearly every row that is simply **one filing**.

Checking was worth it: "nearly unique" and "unique" lead to different counts.


## Select columns

One column name in square brackets gives a **Series**: a single labeled column. A **list** of column names gives a smaller **DataFrame**. Notice the two pairs of brackets in the second line: the outer pair selects, and the inner pair is the list.

**Predict:** What type and shape will each print?


In [ ]:
form_series = filings["Form Type"]
two_columns = filings[["Company Name", "Form Type"]]

print(type(form_series), form_series.shape)
print(type(two_columns), two_columns.shape)


In [ ]:
form_series


`type` names the full Pandas type: the last word, `Series` or `DataFrame`, is the part to read. A Series has a shape with one number, such as `(276622,)`: values and no second dimension, like a list. A DataFrame has rows **and** columns. Keep a DataFrame when the result should still be a table.

### `.loc[rows, columns]`

`.loc` selects by **name**: first which rows, then which columns, separated by a comma. You used `starbucks.loc[2, "Form Type"]` for one value. A colon `:` in the rows position means "all rows," and a list of names in the columns position keeps those columns.

**Predict:** How many rows and columns will this give?


In [56]:
filings.loc[0:20, ["Company Name", "Date Filed"]]


,Company Name,Date Filed
0,OLD MARKET CAPITAL Corp,2025-11-14
1,OLD MARKET CAPITAL Corp,2025-12-22
2,OLD MARKET CAPITAL Corp,2025-10-08
3,OLD MARKET CAPITAL Corp,2025-10-08
4,OLD MARKET CAPITAL Corp,2025-10-08
5,OLD MARKET CAPITAL Corp,2025-10-08
6,OLD MARKET CAPITAL Corp,2025-10-08
7,OLD MARKET CAPITAL Corp,2025-11-26
8,OLD MARKET CAPITAL Corp,2025-12-18
9,OLD MARKET CAPITAL Corp,2025-11-14


## Exercise 1 — Keep three columns

> **Your turn:** Work on your own or with your table before we work through this together.

Use `.loc` to make `filing_columns`: all rows of `filings`, with the columns `CIK`, `Form Type`, and `Date Filed`, in that order. The result must be a DataFrame. Column names are case-sensitive and include the space.


In [61]:
filings

,CIK,Company Name,Form Type,Date Filed,Filename
0,1000045,OLD MARKET CAPITAL Corp,10-Q,2025-11-14,edgar/data/1000045/0001437749-25-035199.txt
1,1000045,OLD MARKET CAPITAL Corp,25,2025-12-22,edgar/data/1000045/0001437749-25-038479.txt
2,1000045,OLD MARKET CAPITAL Corp,4,2025-10-08,edgar/data/1000045/0001398344-25-019020.txt
3,1000045,OLD MARKET CAPITAL Corp,4,2025-10-08,edgar/data/1000045/0001578005-25-000002.txt
4,1000045,OLD MARKET CAPITAL Corp,4,2025-10-08,edgar/data/1000045/0001689627-25-000002.txt
...,...,...,...,...,...
276617,99780,TRINITY INDUSTRIES INC,4,2025-10-02,edgar/data/99780/0000099780-25-000109.txt
276618,99780,TRINITY INDUSTRIES INC,4,2025-10-02,edgar/data/99780/0000099780-25-000110.txt
276619,99780,TRINITY INDUSTRIES INC,4,2025-10-02,edgar/data/99780/0000099780-25-000111.txt
276620,99780,TRINITY INDUSTRIES INC,8-K,2025-10-20,edgar/data/99780/0000099780-25-000114.txt


In [ ]:
filing_columns = filings[['CIK', 'Form Type', 'Date Filed']]

In [63]:
filing_columns = filings.loc[:,['CIK', 'Form Type', 'Date Filed']]

In [64]:
#@title Check your work { display-mode: "form" }
import math

import numpy as np
import pandas as pd

check_tally = {"passed": 0, "total": 0}


class _NotReady(Exception):
    pass


def need(name, kind=None):
    """Return the student's variable, or stop this one check with a friendly reason."""
    if name not in globals():
        raise _NotReady(name + " does not exist yet. Run your work cell first. If it stopped with an "
                        "error, read the last line of the error, fix it, and rerun the work cell.")
    value = globals()[name]
    if value is None:
        raise _NotReady(name + " is still None. Replace the starter None in your work cell.")
    if kind == "DataFrame" and not isinstance(value, pd.DataFrame):
        if isinstance(value, pd.Series):
            raise _NotReady(name + " is a Series (one column), but this exercise needs a DataFrame. "
                            'Put the column names in a list, as in .loc[rows, ["a", "b"]].')
        raise _NotReady(name + " is a " + type(value).__name__ + ", but this exercise needs a DataFrame.")
    return value


def _short(value):
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not math.isnan(value):
        value = round(value, 4)
    text = repr(value)
    return text if len(text) <= 160 else text[:160] + "..."


def _matches(actual, expected):
    if isinstance(expected, np.generic):
        expected = expected.item()
    if isinstance(expected, bool):
        return type(actual).__name__ in ("bool", "bool_") and bool(actual) == expected
    if isinstance(expected, float):
        if type(actual).__name__ in ("bool", "bool_", "str"):
            return False
        number = float(actual)
        if math.isnan(expected):
            return math.isnan(number)
        return not math.isnan(number) and abs(number - expected) <= 0.005
    if isinstance(expected, list):
        if not isinstance(actual, list) or len(actual) != len(expected):
            return False
        return all(_matches(a, e) for a, e in zip(actual, expected))
    if isinstance(expected, int):
        if type(actual).__name__ in ("bool", "bool_", "str"):
            return False
        return bool(actual == expected)
    return bool(actual == expected)


def first_difference(labels, actual, expected):
    """Return "all match", or describe the first label whose value differs."""
    actual, expected = list(actual), list(expected)
    if len(actual) != len(expected):
        return str(len(actual)) + " rows, but expected " + str(len(expected))
    for label, got, want in zip(labels, actual, expected):
        if not _matches(got, want):
            return str(label) + ": got " + _short(got) + ", expected " + _short(want)
    return "all match"


def check(case_name, get_actual, expected, hint=""):
    """Run get_actual() safely and report whether it matches expected (a value or a function)."""
    check_tally["total"] = check_tally["total"] + 1
    try:
        if callable(expected):
            expected = expected()
        actual = get_actual()
    except _NotReady as reason:
        print("NOT YET —", case_name + ":", reason)
        return
    except Exception as error:
        print("NOT YET —", case_name + ": checking raised " + type(error).__name__ + ": " + str(error)[:150])
        fallback = "Read the error, fix your work cell, and rerun it before this check."
        print("    Hint:", fallback if callable(hint) or not hint else hint)
        return
    try:
        matched = _matches(actual, expected)
    except Exception:
        matched = False
    if matched:
        check_tally["passed"] = check_tally["passed"] + 1
        print("✓", case_name, "— passed")
    else:
        print("NOT YET —", case_name + ": expected", _short(expected), "but got", _short(actual))
        if callable(hint):
            hint = hint(actual)
        if hint:
            print("    Hint:", hint)


def check_summary(label):
    """Print how many checks passed."""
    if check_tally["passed"] == check_tally["total"]:
        print(label + ": all " + str(check_tally["total"]) + " checks passed ✓")
    else:
        print(label + ": NOT YET — " + str(check_tally["passed"]) + " of " + str(check_tally["total"])
              + " checks passed. Fix the first NOT YET line, rerun your work cell, then rerun this check.")


def need_index(name="filings"):
    """Return a loaded EDGAR index table, or stop this one check if it has no rows."""
    table = globals().get(name)
    if not isinstance(table, pd.DataFrame) or len(table) == 0:
        raise _NotReady("the EDGAR index is not loaded (" + name + " has no rows). Put your own name and email "
                        "in sec_identity, run that cell, rerun the cell that makes " + name
                        + ", then rerun your work cell.")
    return table


check("Columns, in order", lambda: need("filing_columns", "DataFrame").columns.tolist(),
      ["CIK", "Form Type", "Date Filed"],
      "List the three names inside the brackets, in the order given.")
check("All rows kept", lambda: len(need("filing_columns", "DataFrame")), lambda: len(need_index()),
      "Use : in the rows position to keep every row.")
check("Original table unchanged", lambda: need_index().columns.tolist(),
      ["CIK", "Company Name", "Form Type", "Date Filed", "Filename"],
      "Selecting columns should not change filings. Rerun the cell that makes filings.")
check_summary("Exercise 1")


✓ Columns, in order — passed
✓ All rows kept — passed
✓ Original table unchanged — passed
Exercise 1: all 3 checks passed ✓


## Keep rows with a Boolean mask

A comparison on a column checks **every row at once** and gives a Series of `True` and `False`, one per row. This is called a **Boolean mask**. Here is `filings` again; the cell after it makes a mask that is `True` in every 10-K row.


In [65]:
filings


,CIK,Company Name,Form Type,Date Filed,Filename
0,1000045,OLD MARKET CAPITAL Corp,10-Q,2025-11-14,edgar/data/1000045/0001437749-25-035199.txt
1,1000045,OLD MARKET CAPITAL Corp,25,2025-12-22,edgar/data/1000045/0001437749-25-038479.txt
2,1000045,OLD MARKET CAPITAL Corp,4,2025-10-08,edgar/data/1000045/0001398344-25-019020.txt
3,1000045,OLD MARKET CAPITAL Corp,4,2025-10-08,edgar/data/1000045/0001578005-25-000002.txt
4,1000045,OLD MARKET CAPITAL Corp,4,2025-10-08,edgar/data/1000045/0001689627-25-000002.txt
...,...,...,...,...,...
276617,99780,TRINITY INDUSTRIES INC,4,2025-10-02,edgar/data/99780/0000099780-25-000109.txt
276618,99780,TRINITY INDUSTRIES INC,4,2025-10-02,edgar/data/99780/0000099780-25-000110.txt
276619,99780,TRINITY INDUSTRIES INC,4,2025-10-02,edgar/data/99780/0000099780-25-000111.txt
276620,99780,TRINITY INDUSTRIES INC,8-K,2025-10-20,edgar/data/99780/0000099780-25-000114.txt


In [68]:
10 == 10

True

In [70]:
tenq = filings["Form Type"] == '10-Q'

In [71]:
is_10k = filings["Form Type"] == "10-K"


In [72]:
is_10k

,Form Type
0,False
1,False
2,False
3,False
4,False
...,...
276617,False
276618,False
276619,False
276620,False


In [77]:
filings.loc[filings["Form Type"] == "10-Q"]

,CIK,Company Name,Form Type,Date Filed,Filename
0,1000045,OLD MARKET CAPITAL Corp,10-Q,2025-11-14,edgar/data/1000045/0001437749-25-035199.txt
29,1000209,MEDALLION FINANCIAL CORP,10-Q,2025-11-04,edgar/data/1000209/0001193125-25-264655.txt
32,1000228,HENRY SCHEIN INC,10-Q,2025-11-04,edgar/data/1000228/0001000228-25-000066.txt
1236,1000623,"Mativ Holdings, Inc.",10-Q,2025-11-06,edgar/data/1000623/0001000623-25-000064.txt
1243,1000694,NOVAVAX INC,10-Q,2025-11-06,edgar/data/1000694/0001000694-25-000046.txt
...,...,...,...,...,...
276588,98362,TIMKEN CO,10-Q,2025-10-29,edgar/data/98362/0000098362-25-000142.txt
276594,98677,TOOTSIE ROLL INDUSTRIES INC,10-Q,2025-11-07,edgar/data/98677/0001104659-25-108351.txt
276607,99250,"TRANSCONTINENTAL GAS PIPE LINE COMPANY, LLC",10-Q,2025-11-03,edgar/data/99250/0000107263-25-000145.txt
276610,99302,TRANSCAT INC,10-Q,2025-11-05,edgar/data/99302/0001437749-25-033338.txt


In [80]:
filings

,CIK,Company Name,Form Type,Date Filed,Filename
0,1000045,OLD MARKET CAPITAL Corp,10-Q,2025-11-14,edgar/data/1000045/0001437749-25-035199.txt
1,1000045,OLD MARKET CAPITAL Corp,25,2025-12-22,edgar/data/1000045/0001437749-25-038479.txt
2,1000045,OLD MARKET CAPITAL Corp,4,2025-10-08,edgar/data/1000045/0001398344-25-019020.txt
3,1000045,OLD MARKET CAPITAL Corp,4,2025-10-08,edgar/data/1000045/0001578005-25-000002.txt
4,1000045,OLD MARKET CAPITAL Corp,4,2025-10-08,edgar/data/1000045/0001689627-25-000002.txt
...,...,...,...,...,...
276617,99780,TRINITY INDUSTRIES INC,4,2025-10-02,edgar/data/99780/0000099780-25-000109.txt
276618,99780,TRINITY INDUSTRIES INC,4,2025-10-02,edgar/data/99780/0000099780-25-000110.txt
276619,99780,TRINITY INDUSTRIES INC,4,2025-10-02,edgar/data/99780/0000099780-25-000111.txt
276620,99780,TRINITY INDUSTRIES INC,8-K,2025-10-20,edgar/data/99780/0000099780-25-000114.txt


In [81]:
eightKs = filings.loc[filings["Form Type"] == "8-K"]


In [82]:
eightKs

,CIK,Company Name,Form Type,Date Filed,Filename
9,1000045,OLD MARKET CAPITAL Corp,8-K,2025-11-14,edgar/data/1000045/0001437749-25-035202.txt
10,1000045,OLD MARKET CAPITAL Corp,8-K,2025-12-02,edgar/data/1000045/0001437749-25-036667.txt
11,1000045,OLD MARKET CAPITAL Corp,8-K,2025-12-10,edgar/data/1000045/0001437749-25-037398.txt
12,1000045,OLD MARKET CAPITAL Corp,8-K,2025-12-11,edgar/data/1000045/0001437749-25-037549.txt
30,1000209,MEDALLION FINANCIAL CORP,8-K,2025-10-28,edgar/data/1000209/0001193125-25-253496.txt
...,...,...,...,...,...
276609,99250,"TRANSCONTINENTAL GAS PIPE LINE COMPANY, LLC",8-K,2025-11-20,edgar/data/99250/0001193125-25-289836.txt
276611,99302,TRANSCAT INC,8-K,2025-11-03,edgar/data/99302/0001437749-25-032773.txt
276614,99614,TRI-CONTINENTAL Corp,8-K,2025-11-18,edgar/data/99614/0001193125-25-285571.txt
276620,99780,TRINITY INDUSTRIES INC,8-K,2025-10-20,edgar/data/99780/0000099780-25-000114.txt


The collapsed cell below draws what a mask does, using six real rows of `filings`. Run it, then click the three buttons in order.


In [78]:
#@title Run this cell to see a mask at work { display-mode: "form" }
# Supplied code: draws the picture below. You do not need to read it.
from html import escape
from IPython.display import IFrame, display

_mask_page = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta http-equiv="Content-Security-Policy" content="default-src 'none'; script-src 'unsafe-inline'; style-src 'unsafe-inline'">
<title>A Boolean mask, step by step</title>
<style>
:root { color-scheme: light dark; --fg: #1a1c1f; --muted: #6b7078; --line: #d9dce1; --true: #e3f5e8; --true-fg: #0b7a35;
        --false: #fbe9e7; --false-fg: #b3261e; --accent: #1a73e8; --code: #f3f4f6; }
@media (prefers-color-scheme: dark) {
  :root { --fg: #e8eaed; --muted: #9aa0a6; --line: #3c4043; --true: #16351f; --true-fg: #81c995;
          --false: #3d1c1a; --false-fg: #f28b82; --accent: #8ab4f8; --code: #2a2b2e; }
}
body { margin: 0; padding: 12px 16px; font: 15px/1.45 system-ui, -apple-system, "Segoe UI", Arial, sans-serif; color: var(--fg); }
h2 { font-size: 18px; margin: 0 0 10px; }
.steps { display: flex; gap: 8px; flex-wrap: wrap; margin-bottom: 10px; }
button { font: inherit; padding: 5px 12px; border: 1px solid var(--line); border-radius: 6px; background: transparent; color: var(--fg); cursor: pointer; }
button[aria-pressed="true"] { border-color: var(--accent); color: var(--accent); font-weight: 600; }
pre { margin: 0 0 4px; padding: 6px 10px; background: var(--code); border-radius: 6px; font: 14px/1.4 ui-monospace, Menlo, Consolas, monospace; white-space: pre-wrap; }
.note { color: var(--muted); font-size: 13px; margin: 6px 0 10px; min-height: 2.6em; }
.stage { display: flex; gap: 14px; align-items: flex-start; }
table { border-collapse: collapse; font-size: 14px; }
caption { text-align: left; font: 600 14px ui-monospace, Menlo, Consolas, monospace; padding-bottom: 4px; }
th, td { border-bottom: 1px solid var(--line); padding: 4px 10px; text-align: left; white-space: nowrap; height: 22px; }
th { font-weight: 600; }
td.label { color: var(--muted); text-align: right; }
tr.gap td { color: var(--muted); text-align: center; height: 14px; padding: 0; border-bottom: 0; }
tr { transition: opacity 0.45s ease, background-color 0.3s ease; }
.marked tr.keep td { background: var(--true); }
.marked tr.drop td { background: var(--false); }
tr.leaving { opacity: 0; }
td.bool { font-family: ui-monospace, Menlo, Consolas, monospace; font-weight: 600; }
td.bool.yes { color: var(--true-fg); }
td.bool.no { color: var(--false-fg); }
.hidden { display: none; }
@media (prefers-reduced-motion: reduce) { tr { transition: none; } }
</style>
</head>
<body>
<h2>Keep only the 10-K rows</h2>
<div class="steps">
  <button type="button" data-step="0" aria-pressed="true">filings</button>
  <button type="button" data-step="1" aria-pressed="false">1 · Make the mask</button>
  <button type="button" data-step="2" aria-pressed="false">2 · Use the mask</button>
</div>
<div id="code"></div>
<div class="note" id="note" role="status" aria-live="polite"></div>
<div class="stage">
  <table id="data">
    <caption id="name">filings</caption>
    <thead><tr><th></th><th>Company Name</th><th>Form Type</th><th>Date Filed</th></tr></thead>
    <tbody id="rows"></tbody>
  </table>
  <table id="mask" class="hidden">
    <caption>is_10k</caption>
    <thead><tr><th></th><th>&nbsp;</th></tr></thead>
    <tbody id="maskrows"></tbody>
  </table>
</div>
<script>
(() => {
  const rows = [
    [0, "OLD MARKET CAPITAL Corp", "10-Q", "2025-11-14"],
    [9, "OLD MARKET CAPITAL Corp", "8-K", "2025-11-14"],
    [49, "OPTICAL CABLE CORP", "10-K", "2025-12-18"],
    [1887, "TYSON FOODS, INC.", "10-K", "2025-11-10"],
    [244416, "STARBUCKS CORP", "10-K", "2025-11-14"],
    [244434, "STARBUCKS CORP", "8-K", "2025-10-29"]
  ];
  const code = [
    ["filings"],
    ['is_10k = filings["Form Type"] == "10-K"'],
    ["filings_10k = filings[is_10k]", '# Same rows, with the mask written directly inside the brackets:\nfilings_10k = filings[filings["Form Type"] == "10-K"]']
  ];
  const notes = [
    "Six real rows of filings (the full table has about 277,000). Two of its five columns are left out to save space.",
    "The comparison checks every row at once. is_10k has one True or False per row, with the same row labels as filings.",
    "Only the rows where the mask is True are kept. They keep their original labels (49, 1887, 244416), and filings itself is unchanged."
  ];
  const isKeep = row => row[2] === "10-K";
  const cell = (text, cls) => "<td" + (cls ? ' class="' + cls + '"' : "") + ">" + text + "</td>";
  const gap = cols => '<tr class="gap"><td colspan="' + cols + '">&#8942;</td></tr>';
  const body = document.getElementById("rows");
  const maskBody = document.getElementById("maskrows");
  const stage = document.querySelector(".stage");
  let step = 0;
  let timer = null;

  function draw(next) {
    step = next;
    clearTimeout(timer);
    const shown = step === 2 ? rows.filter(isKeep) : rows;
    body.innerHTML = shown.map((row, i) =>
      (i > 0 ? gap(4) : "") + '<tr class="' + (isKeep(row) ? "keep" : "drop") + '">' +
      cell(row[0], "label") + cell(row[1]) + cell(row[2]) + cell(row[3]) + "</tr>").join("");
    maskBody.innerHTML = rows.map((row, i) =>
      (i > 0 ? gap(2) : "") + "<tr>" + cell(row[0], "label") +
      cell(isKeep(row) ? "True" : "False", "bool " + (isKeep(row) ? "yes" : "no")) + "</tr>").join("");
    document.getElementById("mask").classList.toggle("hidden", step !== 1);
    stage.classList.toggle("marked", step === 1);
    document.getElementById("name").textContent = step === 2 ? "filings_10k" : "filings";
    document.getElementById("code").innerHTML = code[step].map(text =>
      "<pre>" + text.replace(/&/g, "&amp;").replace(/</g, "&lt;") + "</pre>").join("");
    document.getElementById("note").textContent = notes[step];
    document.querySelectorAll("button").forEach(button =>
      button.setAttribute("aria-pressed", String(Number(button.dataset.step) === step)));
  }

  function go(next) {
    if (next === step) return;
    if (next === 2 && step === 1) {
      document.querySelectorAll("#rows tr.drop, #rows tr.gap, #maskrows tr").forEach(row => row.classList.add("leaving"));
      timer = setTimeout(() => draw(2), 500);
      return;
    }
    draw(next);
  }

  document.querySelectorAll("button").forEach(button =>
    button.addEventListener("click", () => go(Number(button.dataset.step))));
  draw(0);
})();
</script>
</body>
</html>
"""

display(IFrame(
    src="about:blank",
    width="100%",
    height=600,
    extras=[
        'title="A Boolean mask, step by step"',
        'sandbox="allow-scripts"',
        'style="border:0;display:block;"',
        'srcdoc="' + escape(_mask_page, quote=True) + '"',
    ],
))


In code, the mask goes in the rows position of `.loc`: `filings.loc[is_10k, ...]` keeps the rows where `is_10k` is `True`, and the list after the comma keeps those columns. Then `len` counts the rows that were kept.

**Predict:** About how many of the quarter's filings are annual reports (`10-K`): tens, hundreds, or thousands?


In [ ]:
filings_10k = filings.loc[is_10k, ["Company Name", "Form Type", "Date Filed"]]
print("Rows kept:", len(filings_10k))
filings_10k


Most companies end their fiscal year in December and file the 10-K early the next year, so October to December has only a few hundred.

As in the picture, the kept rows keep their original labels; nothing was renumbered or deleted from `filings`. Now labels and positions differ: `filings_10k.iloc[0:3]` gives the rows in **positions** 0, 1, and 2, whatever their labels. Like a list slice, it stops before position 3.


In [ ]:
filings_10k.iloc[0:3]


### A named mask, or a mask written in place

A mask does not need a name. You can write the comparison directly inside the brackets, where `is_10k` went. And when you want every column, square brackets alone are enough: `filings[is_10k]` keeps rows. (Square brackets keep **columns** when you give them column names, as in `filings[["Company Name", "Form Type"]]`, and **rows** when you give them a mask.)

**Predict:** Will these three lines print the same number?


In [ ]:
print(len(filings.loc[is_10k]))
print(len(filings[is_10k]))
print(len(filings[filings["Form Type"] == "10-K"]))


All three keep the same rows. Name a mask when you want to look at it or use it more than once; write it in place when it is short. With two conditions, writing them in place is common.


### Two conditions: `&` and `|`

To require **both** conditions, join two masks with `&`. To accept **either**, use `|` (the vertical bar, usually Shift + backslash). Put **each comparison in its own parentheses**.

Python's `and` and `or` from Notebook 4 work on one `True` or `False` at a time. A mask holds many, so Pandas uses `&` and `|` instead.

**Predict:** How many rows will be left after this mask? Starbucks's fiscal year ends in late September or early October.


and : &

or : |

In [83]:
starbucks_10k = filings.loc[(filings["CIK"] == 829224) & (filings["Form Type"] == "10-K")]
starbucks_10k


,CIK,Company Name,Form Type,Date Filed,Filename
244416,829224,STARBUCKS CORP,10-K,2025-11-14,edgar/data/829224/0000829224-25-000114.txt


Splitting a long `.loc[...]` across lines is fine. Python keeps reading until the closing bracket.

**Predict:** This cell uses `|` instead of `&`. How many rows will it keep, and why?


In [84]:
starbucks_or_10k = filings.loc[(filings["CIK"] == 829224) | (filings["Form Type"] == "10-K")]
print("Rows kept:", len(starbucks_or_10k))


Rows kept: 383


In [85]:
starbucks_or_10k

,CIK,Company Name,Form Type,Date Filed,Filename
49,1000230,OPTICAL CABLE CORP,10-K,2025-12-18,edgar/data/1000230/0001437749-25-038228.txt
1304,1001115,GEOSPACE TECHNOLOGIES CORP,10-K,2025-11-21,edgar/data/1001115/0001437749-25-036009.txt
1442,1001601,"MGT CAPITAL INVESTMENTS, INC.",10-K,2025-11-10,edgar/data/1001601/0001493152-25-021523.txt
1568,1002590,"STAR GROUP, L.P.",10-K,2025-12-09,edgar/data/1002590/0001193125-25-311490.txt
1628,1003078,MSC INDUSTRIAL DIRECT CO INC,10-K,2025-10-23,edgar/data/1003078/0001003078-25-000123.txt
...,...,...,...,...,...
271442,928465,AMCON DISTRIBUTING CO,10-K,2025-11-07,edgar/data/928465/0001104659-25-108504.txt
271859,932021,GLOBAL TECHNOLOGIES LTD,10-K,2025-12-30,edgar/data/932021/0001493152-25-029524.txt
272355,933974,"Azenta, Inc.",10-K,2025-12-04,edgar/data/933974/0001437749-25-036931.txt
272711,936395,CIENA CORP,10-K,2025-12-12,edgar/data/936395/0001628280-25-056698.txt


In [87]:
8 == 8 or 9 == 9

True

Hundreds of rows: every Starbucks filing of any form, plus every company's 10-K. The code ran without an error, but it answered a different question. A filter that runs is not the same as a filter that is right, so always check the rows it kept.


## Exercise 2 — Starbucks's current reports

> **Your turn:** Work on your own or with your table before we work through this together.

Make `starbucks_8k`: the rows of `filings` with CIK `829224` **and** form type `"8-K"`. Keep the columns `Company Name`, `Form Type`, and `Date Filed`, in that order.

Before running, look back at the three typed rows at the top of the notebook. How many rows do you expect?


In [ ]:
#@title Check your work { display-mode: "form" }
import math

import numpy as np
import pandas as pd

check_tally = {"passed": 0, "total": 0}


class _NotReady(Exception):
    pass


def need(name, kind=None):
    """Return the student's variable, or stop this one check with a friendly reason."""
    if name not in globals():
        raise _NotReady(name + " does not exist yet. Run your work cell first. If it stopped with an "
                        "error, read the last line of the error, fix it, and rerun the work cell.")
    value = globals()[name]
    if value is None:
        raise _NotReady(name + " is still None. Replace the starter None in your work cell.")
    if kind == "DataFrame" and not isinstance(value, pd.DataFrame):
        if isinstance(value, pd.Series):
            raise _NotReady(name + " is a Series (one column), but this exercise needs a DataFrame. "
                            'Put the column names in a list, as in .loc[rows, ["a", "b"]].')
        raise _NotReady(name + " is a " + type(value).__name__ + ", but this exercise needs a DataFrame.")
    return value


def _short(value):
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not math.isnan(value):
        value = round(value, 4)
    text = repr(value)
    return text if len(text) <= 160 else text[:160] + "..."


def _matches(actual, expected):
    if isinstance(expected, np.generic):
        expected = expected.item()
    if isinstance(expected, bool):
        return type(actual).__name__ in ("bool", "bool_") and bool(actual) == expected
    if isinstance(expected, float):
        if type(actual).__name__ in ("bool", "bool_", "str"):
            return False
        number = float(actual)
        if math.isnan(expected):
            return math.isnan(number)
        return not math.isnan(number) and abs(number - expected) <= 0.005
    if isinstance(expected, list):
        if not isinstance(actual, list) or len(actual) != len(expected):
            return False
        return all(_matches(a, e) for a, e in zip(actual, expected))
    if isinstance(expected, int):
        if type(actual).__name__ in ("bool", "bool_", "str"):
            return False
        return bool(actual == expected)
    return bool(actual == expected)


def first_difference(labels, actual, expected):
    """Return "all match", or describe the first label whose value differs."""
    actual, expected = list(actual), list(expected)
    if len(actual) != len(expected):
        return str(len(actual)) + " rows, but expected " + str(len(expected))
    for label, got, want in zip(labels, actual, expected):
        if not _matches(got, want):
            return str(label) + ": got " + _short(got) + ", expected " + _short(want)
    return "all match"


def check(case_name, get_actual, expected, hint=""):
    """Run get_actual() safely and report whether it matches expected (a value or a function)."""
    check_tally["total"] = check_tally["total"] + 1
    try:
        if callable(expected):
            expected = expected()
        actual = get_actual()
    except _NotReady as reason:
        print("NOT YET —", case_name + ":", reason)
        return
    except Exception as error:
        print("NOT YET —", case_name + ": checking raised " + type(error).__name__ + ": " + str(error)[:150])
        fallback = "Read the error, fix your work cell, and rerun it before this check."
        print("    Hint:", fallback if callable(hint) or not hint else hint)
        return
    try:
        matched = _matches(actual, expected)
    except Exception:
        matched = False
    if matched:
        check_tally["passed"] = check_tally["passed"] + 1
        print("✓", case_name, "— passed")
    else:
        print("NOT YET —", case_name + ": expected", _short(expected), "but got", _short(actual))
        if callable(hint):
            hint = hint(actual)
        if hint:
            print("    Hint:", hint)


def check_summary(label):
    """Print how many checks passed."""
    if check_tally["passed"] == check_tally["total"]:
        print(label + ": all " + str(check_tally["total"]) + " checks passed ✓")
    else:
        print(label + ": NOT YET — " + str(check_tally["passed"]) + " of " + str(check_tally["total"])
              + " checks passed. Fix the first NOT YET line, rerun your work cell, then rerun this check.")


def need_index(name="filings"):
    """Return a loaded EDGAR index table, or stop this one check if it has no rows."""
    table = globals().get(name)
    if not isinstance(table, pd.DataFrame) or len(table) == 0:
        raise _NotReady("the EDGAR index is not loaded (" + name + " has no rows). Put your own name and email "
                        "in sec_identity, run that cell, rerun the cell that makes " + name
                        + ", then rerun your work cell.")
    return table


def _expected_rows():
    table = need_index()
    return int(((table["CIK"] == 829224) & (table["Form Type"] == "8-K")).sum())


def _rows_hint(got):
    if got == 0:
        return ("No rows: CIK is a number in this table, so compare it with 829224 without quotes, "
                "and join the two conditions with &.")
    if got > 10:
        return ("Too many rows: you used | (either) instead of & (both), or left out one condition. "
                "Put each comparison in its own parentheses.")
    return '== keeps only the exact text "8-K". Form "8-K/A" (an amended 8-K) is a different form.'


def _only_starbucks_8k():
    kept = need("starbucks_8k", "DataFrame")
    table = need_index()
    return bool(table.loc[kept.index, "CIK"].eq(829224).all()
                and table.loc[kept.index, "Form Type"].eq("8-K").all())


check("Columns, in order", lambda: need("starbucks_8k", "DataFrame").columns.tolist(),
      ["Company Name", "Form Type", "Date Filed"],
      "Keep exactly these three columns, in this order.")
check("Rows kept", lambda: len(need("starbucks_8k", "DataFrame")), _expected_rows, _rows_hint)
check("Only Starbucks 8-K rows", _only_starbucks_8k, lambda: (need_index(), True)[1],
      'Use (filings["CIK"] == 829224) & (filings["Form Type"] == "8-K").')
check_summary("Exercise 2")


### Now include amendments

The index lists a third Starbucks report with form `8-K/A`, an amendment. `==` compares the whole text, so it is not an `8-K` row. Whether to count amendments is a choice you should make and say out loud.

To count them, write the "8-K or 8-K/A" part with `|`, wrap it in its own parentheses, then add the CIK condition with `&`. **Predict:** How many rows will this keep?


In [ ]:
starbucks_current = filings.loc[
    (filings["CIK"] == 829224)
    & ((filings["Form Type"] == "8-K") | (filings["Form Type"] == "8-K/A")),
    ["Company Name", "Form Type", "Date Filed"],
]
starbucks_current


Three rows: the two 8-Ks and the 8-K/A. The extra parentheses matter. Without them, Python does `&` before `|`, as it does `*` before `+`, so the CIK condition would apply only to the 8-K part, and every company's 8-K/A would be kept too.


## Work on a copy

In Notebook 5, two names could point to the **same list**, so changing it through one name changed it for both. DataFrames behave the same way. Assigning a new column with `table["new_name"] = value` changes that table.

**Predict:** After the second line, does `filings` have a `checked` column?


In [ ]:
working = filings
working["checked"] = "no"
print("filings columns:", filings.columns.tolist())

filings = filings.drop(columns="checked")
print("After drop:", filings.columns.tolist())


In [ ]:
filings


`working = filings` did not make a second table; it gave the same table a second name. So the new column appeared in `filings` too. The last two lines remove it again: `.drop(columns="checked")` returns the table without that column, and we assign the result back to `filings`.

**Keeping rows is different.** The next cell keeps the 10-Q rows and adds a `flag` column to the result. **Predict:** After it runs, will `filings` have a `flag` column? Then run the three cells and compare the two tables.


In [ ]:
quarterly = filings.loc[filings["Form Type"] == "10-Q"]
quarterly["flag"] = "review"


In [ ]:
quarterly


In [ ]:
filings


`quarterly` has the `flag` column and `filings` does not. Keeping rows with a mask builds a **new table** that holds a copy of the kept rows, so a change to `quarterly` cannot reach `filings`. Compare `working = filings` above, which built nothing new.

So what is the **SettingWithCopyWarning** you may have seen ("A value is trying to be set on a copy of a slice from a DataFrame")? Some other ways of selecting can give a **view** instead of a copy: a window onto the original table's values, so that a change through the view could reach the original. Pandas cannot always tell which one you have, so when you change a selection it warns you. Here the warning means: *this change stays in `quarterly`; `filings` will not change.* It is a reminder, not an error.

End the selection with **`.copy()`** whenever you plan to change the result. It says you want a separate table, so Pandas has nothing to warn about:

```python
quarterly = filings.loc[filings["Form Type"] == "10-Q"].copy()
quarterly["flag"] = "review"
```

Whether the warning appears depends on how you selected and on the Pandas version, so no warning does not prove the code does what you meant. Write `.copy()` anyway.


## Many quarters at once

One quarter is one file. To study many years, put one loop inside another: the outer loop visits each year, and for each year the inner loop visits quarters 1 to 4. Each pass downloads one file, reads it into a table, and adds the table to a list. Then **`pd.concat`** stacks a list of tables with the same columns into one table:

```python
combined = pd.concat([table_a, table_b], ignore_index=True)
```

Each stacked row keeps its old label, so the labels 0, 1, 2, ... would repeat once per file. `ignore_index=True` numbers the rows again from 0.

Inside the loops, use new names such as `quarter_response` and `quarter_table`. Reusing `response` or `filings` would replace the 2025 Q4 data the rest of this notebook uses.


## Exercise 3 — Eleven years of filings

> **Your turn:** Work on your own or with your table before we work through this together.

1. Loop over the years 2015 to 2025 and, inside that, the quarters 1 to 4 (44 files). For each one, download `master.idx` with the same User-Agent as before, save it as `master_2015_Q1.idx`, `master_2015_Q2.idx`, and so on, read it with `pd.read_csv` (the same `sep` and `skiprows`), and append the table to a list. Build each URL with an f-string: the 2025 Q4 URL from earlier, with the year and the quarter number filled in.
2. Stack the list into `combined_index` with `pd.concat(..., ignore_index=True)`.
3. Make `sbux_10k`: the rows of `combined_index` with CIK `829224` (Starbucks) **and** form type `"10-K"`.

**Predict:** How many rows will `sbux_10k` have?


In [ ]:
#@title Check your work { display-mode: "form" }
import math

import numpy as np
import pandas as pd

check_tally = {"passed": 0, "total": 0}


class _NotReady(Exception):
    pass


def need(name, kind=None):
    """Return the student's variable, or stop this one check with a friendly reason."""
    if name not in globals():
        raise _NotReady(name + " does not exist yet. Run your work cell first. If it stopped with an "
                        "error, read the last line of the error, fix it, and rerun the work cell.")
    value = globals()[name]
    if value is None:
        raise _NotReady(name + " is still None. Replace the starter None in your work cell.")
    if kind == "DataFrame" and not isinstance(value, pd.DataFrame):
        if isinstance(value, pd.Series):
            raise _NotReady(name + " is a Series (one column), but this exercise needs a DataFrame. "
                            'Put the column names in a list, as in .loc[rows, ["a", "b"]].')
        raise _NotReady(name + " is a " + type(value).__name__ + ", but this exercise needs a DataFrame.")
    return value


def _short(value):
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not math.isnan(value):
        value = round(value, 4)
    text = repr(value)
    return text if len(text) <= 160 else text[:160] + "..."


def _matches(actual, expected):
    if isinstance(expected, np.generic):
        expected = expected.item()
    if isinstance(expected, bool):
        return type(actual).__name__ in ("bool", "bool_") and bool(actual) == expected
    if isinstance(expected, float):
        if type(actual).__name__ in ("bool", "bool_", "str"):
            return False
        number = float(actual)
        if math.isnan(expected):
            return math.isnan(number)
        return not math.isnan(number) and abs(number - expected) <= 0.005
    if isinstance(expected, list):
        if not isinstance(actual, list) or len(actual) != len(expected):
            return False
        return all(_matches(a, e) for a, e in zip(actual, expected))
    if isinstance(expected, int):
        if type(actual).__name__ in ("bool", "bool_", "str"):
            return False
        return bool(actual == expected)
    return bool(actual == expected)


def first_difference(labels, actual, expected):
    """Return "all match", or describe the first label whose value differs."""
    actual, expected = list(actual), list(expected)
    if len(actual) != len(expected):
        return str(len(actual)) + " rows, but expected " + str(len(expected))
    for label, got, want in zip(labels, actual, expected):
        if not _matches(got, want):
            return str(label) + ": got " + _short(got) + ", expected " + _short(want)
    return "all match"


def check(case_name, get_actual, expected, hint=""):
    """Run get_actual() safely and report whether it matches expected (a value or a function)."""
    check_tally["total"] = check_tally["total"] + 1
    try:
        if callable(expected):
            expected = expected()
        actual = get_actual()
    except _NotReady as reason:
        print("NOT YET —", case_name + ":", reason)
        return
    except Exception as error:
        print("NOT YET —", case_name + ": checking raised " + type(error).__name__ + ": " + str(error)[:150])
        fallback = "Read the error, fix your work cell, and rerun it before this check."
        print("    Hint:", fallback if callable(hint) or not hint else hint)
        return
    try:
        matched = _matches(actual, expected)
    except Exception:
        matched = False
    if matched:
        check_tally["passed"] = check_tally["passed"] + 1
        print("✓", case_name, "— passed")
    else:
        print("NOT YET —", case_name + ": expected", _short(expected), "but got", _short(actual))
        if callable(hint):
            hint = hint(actual)
        if hint:
            print("    Hint:", hint)


def check_summary(label):
    """Print how many checks passed."""
    if check_tally["passed"] == check_tally["total"]:
        print(label + ": all " + str(check_tally["total"]) + " checks passed ✓")
    else:
        print(label + ": NOT YET — " + str(check_tally["passed"]) + " of " + str(check_tally["total"])
              + " checks passed. Fix the first NOT YET line, rerun your work cell, then rerun this check.")


def need_index(name="filings"):
    """Return a loaded EDGAR index table, or stop this one check if it has no rows."""
    table = globals().get(name)
    if not isinstance(table, pd.DataFrame) or len(table) == 0:
        raise _NotReady("the EDGAR index is not loaded (" + name + " has no rows). Put your own name and email "
                        "in sec_identity, run that cell, rerun the cell that makes " + name
                        + ", then rerun your work cell.")
    return table


def _quarters(table):
    dates = pd.to_datetime(pd.Series(table["Date Filed"].unique()))
    return sorted(dates.dt.to_period("Q").astype(str).unique())


def _saved_rows():
    need("combined_index", "DataFrame")
    total = 0
    for y in range(2015, 2026):
        for q in range(1, 5):
            name = f"master_{y}_Q{q}.idx"
            if not Path(name).is_file():
                raise _NotReady(name + " is not saved. Save each file under that name, then rerun your work cell.")
            total += len(pd.read_csv(name, sep="|", skiprows=[0, 1, 2, 3, 4, 5, 6, 7, 8, 10], usecols=["CIK"]))
    return total


def _only_sbux_10k():
    kept = need("sbux_10k", "DataFrame")
    table = need("combined_index", "DataFrame")
    return bool(table.loc[kept.index, "CIK"].eq(829224).all()
                and table.loc[kept.index, "Form Type"].eq("10-K").all())


from pathlib import Path

check("Every quarter from 2015 to 2025", lambda: _quarters(need("combined_index", "DataFrame")),
      [f"{y}Q{q}" for y in range(2015, 2026) for q in range(1, 5)],
      "Loop over range(2015, 2026) and, inside it, range(1, 5); put {y} and QTR{q} in the URL "
      "and append each table to the list.")
check("Rows of all 44 files", lambda: len(need("combined_index", "DataFrame")), _saved_rows,
      "Each file should appear once: make the empty list before both loops and call pd.concat after them.")
check("Labels run from 0", lambda: bool(need("combined_index", "DataFrame").index.equals(
          pd.RangeIndex(len(need("combined_index", "DataFrame"))))), True,
      "Pass ignore_index=True to pd.concat.")
check("Starbucks's 10-K rows", lambda: len(need("sbux_10k", "DataFrame")),
      lambda: int(((need("combined_index", "DataFrame")["CIK"] == 829224)
                   & (need("combined_index", "DataFrame")["Form Type"] == "10-K")).sum()),
      'Use (combined_index["CIK"] == 829224) & (combined_index["Form Type"] == "10-K"). CIK is a number.')
check("Only Starbucks 10-K rows", _only_sbux_10k, True,
      "Keep the rows of combined_index where both conditions are true.")
check("filings still holds 2025 Q4", lambda: _quarters(need_index()), ["2025Q4"],
      "Something in the loops replaced filings. Use new names inside the loops, rerun from the download cell, "
      "then rerun your work cell.")
check_summary("Exercise 3")


### Your turn, ungraded: search Starbucks's 10-Ks

This part has no check. Bring in your Homework 3 EdgarTools work. Each row of `sbux_10k` is one 10-K, and its accession number is the end of its `Filename`, sliced as before with `[-24:-4]`. Loop over the rows, fetch each 10-K with `edgar.get_by_accession_number`, get its text with `.document.markdown()`, and count how often a keyword of your choice appears. Use new names, such as `tenk` and `tenk_text`, so `filing` and `text` stay as they are.

This is the kind of work Homework 4 asks for, so this is a head start on Homework 4.


In [91]:
qtr = 1
yr = 2024

for yr in range(2010,2025):
    for qtr in range(1,5):
        print(f"https://www.sec.gov/Archives/edgar/full-index/{yr}/QTR{qtr}/master.idx")

https://www.sec.gov/Archives/edgar/full-index/2010/QTR1/master.idx
https://www.sec.gov/Archives/edgar/full-index/2010/QTR2/master.idx
https://www.sec.gov/Archives/edgar/full-index/2010/QTR3/master.idx
https://www.sec.gov/Archives/edgar/full-index/2010/QTR4/master.idx
https://www.sec.gov/Archives/edgar/full-index/2011/QTR1/master.idx
https://www.sec.gov/Archives/edgar/full-index/2011/QTR2/master.idx
https://www.sec.gov/Archives/edgar/full-index/2011/QTR3/master.idx
https://www.sec.gov/Archives/edgar/full-index/2011/QTR4/master.idx
https://www.sec.gov/Archives/edgar/full-index/2012/QTR1/master.idx
https://www.sec.gov/Archives/edgar/full-index/2012/QTR2/master.idx
https://www.sec.gov/Archives/edgar/full-index/2012/QTR3/master.idx
https://www.sec.gov/Archives/edgar/full-index/2012/QTR4/master.idx
https://www.sec.gov/Archives/edgar/full-index/2013/QTR1/master.idx
https://www.sec.gov/Archives/edgar/full-index/2013/QTR2/master.idx
https://www.sec.gov/Archives/edgar/full-index/2013/QTR3/master

In [2]:
sec_identity = ""
if sec_identity == "":
    name = input("Name: ")
    email = input("Email: ")
    sec_identity = name + " " + email
edgar.set_identity(sec_identity)
print("EDGAR identity set.")

Name: nick
Email: nicholashallman@utexas.edu
EDGAR identity set.


In [3]:
url = "https://www.sec.gov/Archives/edgar/full-index/2025/QTR4/master.idx"

In [4]:
response = requests.get(url, headers={"User-Agent": sec_identity}, timeout=60)

In [5]:
local_filename = f"master_2025_Q4.idx"

if response.status_code == 200:
    f = open(local_filename, "wb")
    f.write(response.content)
    f.close()
    print("Saved", local_filename)
else:
    print("Not saved: EDGAR answered with status code", response.status_code)
    print("What to do: check sec_identity (your name and email), rerun that cell, then rerun from the download cell.")

Saved master_2025_Q4.idx


In [6]:
filings = pd.read_csv(f"master_2025_Q4.idx", sep="|", skiprows = [0,1,2,3,4,5,6,7,8,10])

In [ ]:
filings

## Spot the error

Each cell below has one mistake, and each is commented out, so **Run all** skips it. **Predict** what is wrong in each before you try it.

To see the error, select all the lines in a cell and press **Ctrl + /** (**Cmd + /** on a Mac) to remove the `#` marks, run the cell, and read the **last line** of the error message first. Then put the `#` marks back the same way.


In [ ]:
# # Error 1
# filings.loc[(filings["CIK"] == 829224) and (filings["Form Type"] == "8-K")]


In [ ]:
# # Error 2
# filings.loc[filings["CIK"] == 829224 & filings["Form Type"] == "8-K"]


In [ ]:
# # Error 3
# filings.loc[:, ["CIK", "form type"]]


After you have tried all three, compare with this table.

| Error (last line) | Usual cause | Fix |
|---|---|---|
| `ValueError: The truth value of a Series is ambiguous` | `and` or `or` used with masks | Use `&` for both, or the vertical bar for either |
| `TypeError: Cannot perform 'rand_' ...` | No parentheses, so `&` runs before `==` | Put each comparison in its own parentheses |
| `KeyError: "['form type'] not in index"` | A column name is misspelled; names are case-sensitive | Copy the name from `filings.columns.tolist()` |

And two problems that show **no** error: using `|` when you meant `&`, and comparing the number column `CIK` with the text `"829224"` (in quotes), which quietly keeps 0 rows. Check the rows you kept.


## Retrieval

Answer on paper, without running code:

1. `filings[["Form Type"]]` and `filings["Form Type"]`: which is a DataFrame, and what is the other?
2. Why does `filings.loc[filings["CIK"] == "829224"]` keep no rows, and what one change fixes it?
3. Why might `subset = filings.loc[mask]` followed by `subset["new"] = 1` cause a warning, and what one change prevents it?
4. In Exercise 3, how many times does the inner loop's body run in total, why do the loops use `quarter_response` instead of `response`, and what would the labels of `combined_index` look like without `ignore_index=True`?

## Recap

- **Grain** first: say what one row represents before you calculate, and check an ID with `.is_unique`.
- A public file can be downloaded with one request that carries your identity, saved to disk, and read with `pd.read_csv`.
- `table[["a", "b"]]` and `.loc[rows, ["a", "b"]]` select columns and keep a DataFrame; `table["a"]` gives a Series. `.loc` selects by label, `.iloc` by position.
- A **mask** keeps rows, either named or written in place. Join conditions with `&` (both) or `|` (either), each comparison in parentheses. Compare an ID with the same type it has in the table.
- `.copy()` when you keep rows and plan to change columns.
- Nested `for` loops (years, then quarters) can download and read one file per quarter; `pd.concat(list_of_tables, ignore_index=True)` stacks them into one table.

Before finishing, use **Runtime → Restart session and run all**. Colab asks for your name and email again (unless you typed them into `sec_identity`) and downloads all the index files again. Your three Exercise checks should pass.
